In [1]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

load_dotenv()  # expects GOOGLE_API_KEY in your .env

# 1. Sample Data
docs = [
    "Product SKU-99217 is an M2 MacBook Pro laptop with 16GB RAM meant for developers.",
    "MacBook laptops are excellent portable computers for college students and professionals.",
    "Error code ERR-404 represents a page not found server-side issue.",
    "When a web page cannot be located on a server, a HTTP 404 missing status is thrown."
]


c:\Users\asoha\Desktop\cse\langchain_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. Dense retriever (Gemini embeddings + FAISS)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = FAISS.from_texts(docs, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})  # k=4: rank everything, RRF needs ranks not just top-k

In [4]:
# 3. Sparse retriever (BM25, keyword-based)
sparse_retriever = BM25Retriever.from_texts(docs)
sparse_retriever.k = 4

In [5]:
# 4. Manual RRF fusion
# Textbook formula (Cormack, Clarke, Buettcher 2009): score(d) = sum over retrievers of 1 / (k_const + rank(d))
def rrf_fuse(query: str, k_const: int = 60, top_n: int = 2):
    dense_results = dense_retriever.invoke(query)
    sparse_results = sparse_retriever.invoke(query)

    scores = {}     # page_content -> fused RRF score
    doc_lookup = {}  # page_content -> Document object

    for rank, doc in enumerate(dense_results):
        scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (k_const + rank + 1)
        doc_lookup[doc.page_content] = doc

    for rank, doc in enumerate(sparse_results):
        scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (k_const + rank + 1)
        doc_lookup[doc.page_content] = doc

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_lookup[content] for content, _ in ranked[:top_n]]

hybrid_retriever = RunnableLambda(rrf_fuse)

In [6]:
# 5. LLM + prompt
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}
Question: {question}
Answer:
""")

def format_docs(retrieved_documents):
    return "\n\n".join(d.page_content for d in retrieved_documents)

In [7]:
# 6. LCEL pipeline
rag_chain = (
    {
        "context": hybrid_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 7. Execution
if __name__ == "__main__":
    print("--- Query 1: Exact Match (BM25 should win) ---")
    print(rag_chain.invoke("What is SKU-99217?"))

    print("\n--- Query 2: Semantic Meaning (dense should win) ---")
    print(rag_chain.invoke("What happens if a website page is missing?"))

--- Query 1: Exact Match (BM25 should win) ---
SKU-99217 is an M2 MacBook Pro laptop with 16GB RAM meant for developers.

--- Query 2: Semantic Meaning (dense should win) ---
When a web page cannot be located on a server, a HTTP 404 missing status is thrown, and error code ERR-404 represents the page not found server-side issue.
